In [2]:
from py2neo import Graph

graph = Graph(
    "bolt://26.84.188.242:7687",
    auth=("neo4j", "123456789")
)

print(graph.run("RETURN 1").data())

[{'1': 1}]


In [25]:
from py2neo import Graph

# Chi nhánh 1
cn1 = Graph(
    "bolt://26.84.188.242:7687",
    auth=("neo4j", "123456789")
)

# Chi nhánh 2
cn2 = Graph(
    "bolt://26.7.224.249:7687",
    auth=("neo4j", "phthyngx")
)

print(cn1.run("RETURN 'CN1 OK' AS msg").data())
print(cn2.run("RETURN 'CN2 OK' AS msg").data())

[{'msg': 'CN1 OK'}]
[{'msg': 'CN2 OK'}]


In [26]:
from py2neo import Graph

# Kết nối
graph1 = Graph(uri="bolt://26.84.188.242:7687", auth=("neo4j", "123456789"))
graph2 = Graph(uri="bolt://26.7.224.249:7687", auth=("neo4j", "phthyngx"))

# Kiểm tra kết nối bằng truy vấn đơn giản
try:
    result = graph1.run("RETURN 1 AS test").data()
    print("✅ Kết nối thành công! Kết quả test:", result)

    result = graph2.run("RETURN 2 AS test").data()
    print("✅ Kết nối thành công! Kết quả test:", result)

except Exception as e:
    print("❌ Kết nối thất bại. Lỗi:", e)

✅ Kết nối thành công! Kết quả test: [{'test': 1}]
✅ Kết nối thành công! Kết quả test: [{'test': 2}]


In [3]:
from py2neo import Graph
import pandas as pd

graph = Graph(
    "bolt://26.84.188.242:7687",
    auth=("neo4j", "123456789")
)

In [30]:
df = pd.read_csv("DanhMuc_SanPham_Convenience.csv")
for _, row in df.iterrows():

    graph.run("""
        MERGE (d:DanhMuc {
            maDanhMuc:$ma
        })

        SET d.tenDanhMuc = $ten
    """,

    ma=row["MaDanhMuc"],
    ten=row["TenDanhMuc"])

In [36]:
df = pd.read_csv("KhachHang.csv")

for _, row in df.iterrows():

    graph.run("""
        MERGE (k:KhachHang {
            maKhachHang:$ma
        })

        SET k.hoTen = $ten,
            k.sdt = $sdt,
            k.gioiTinh = $gioiTinh,
            k.ngaysinh = $ngaySinh,
            k.diaChi = $diaChi,
            k.diemTichLuy = $diem
    """,

    ma=row["MAKH"],
    ten=row["HOTEN"],
    sdt=row["SDT"],
    ngaySinh=row["NGAYSINH"],
    gioiTinh=row["GIOITINH"],
    diaChi=row["DIACHI"],
    diem=row["DIEMTICHLUY"])

print("Import KhachHang thành công")

Import KhachHang thành công


In [37]:
df = pd.read_csv("SanPham_300.csv")

for _, row in df.iterrows():

    graph.run("""

        MERGE (s:SanPham {
            maSanPham:$maSP
        })

        SET s.tenSanPham = $tenSP,
            s.dvt = $dvt,
            s.giaBan = $giaBan,
            s.giaVon = $giaVon

        WITH s

        MATCH (d:DanhMuc {
            maDanhMuc:$maDM
        })

        MERGE (s)-[:THUOC_DANHMUC]->(d)

    """,

    maSP=row["MaSanPham"],
    tenSP=row["TenSanPham"],
    dvt=row["DVT"],
    giaBan=row["GiaBan"],
    giaVon=row["GiaNhap"],
    maDM=row["MaDanhMuc"])

print("Import SanPham thành công")

Import SanPham thành công


In [39]:
df = pd.read_csv("kho.csv")

for _, row in df.iterrows():

    graph.run("""

        MERGE (k:Kho {
            maKho:$maKho
        })

        SET k.soLuong = $soLuong,
            k.viTri = $viTri

        WITH k

        MATCH (s:SanPham {
            maSanPham:$maSP
        })

        MERGE (k)-[:LUU_TRU]->(s)

    """,

    maKho=row["MaKho"],
    soLuong=row["SoLuong"],
    viTri=row["ViTri"],
    maSP=row["MaSanPham"],
    ngayCapNhat=row["NgayCapNhat"])

print("Import Kho thành công")

Import Kho thành công


In [41]:
df = pd.read_csv("TonKho_CH1.csv")

for _, row in df.iterrows():

    graph.run("""

        MATCH (c:CuaHang {
            maCuaHang:$maCH
        })

        MATCH (s:SanPham {
            maSanPham:$maSP
        })

        MERGE (c)-[r:TON_KHO]->(s)

        SET r.soLuongTrenKe = $slKe,
            r.soLuongTrongKho = $slKho

    """,

    maCH=row["MACH"],
    maSP=row["MaSanPham"],
    slKe=row["SoLuongTrenKe"],
    slKho=row["SoLuongTrongKho"],
    ngayCapNhat=row["NgayCapNhat"])

print("Import TonKho_CuaHang thành công")

Import TonKho_CuaHang thành công


In [42]:
df = pd.read_csv("NhanVien_CH1.csv")

for _, row in df.iterrows():

    graph.run("""

        MERGE (n:NhanVien {
            maNhanVien:$maNV
        })

        SET n.hoTen = $hoTen,
            n.gioiTinh = $gioiTinh,
            n.ngaySinh = $ngaySinh,
            n.chucVu = $chucVu,
            n.luong = $luong,
            n.ngayVaoLam = $ngayVaoLam

        WITH n

        MATCH (c:CuaHang {
            maCuaHang:$maCH
        })

        MERGE (n)-[:LAM_VIEC_TAI]->(c)

    """,

    maNV=row["MANV"],
    maCH=row["MACH"],
    hoTen=row["HOTEN"],
    gioiTinh=row["GIOITINH"],
    ngaySinh=str(row["NGAYSINH"]),
    chucVu=row["CHUCVU"],
    luong=row["LUONG"],
    ngayVaoLam=str(row["NGAYVAOLAM"])
    )

print("Import NhanVien thành công")

Import NhanVien thành công


In [ ]:
df = pd.read_csv("HoaDon_CH1.csv")

for _, row in df.iterrows():

    graph.run("""

        MERGE (h:HoaDon {
            maHoaDon:$maHD
        })

        SET h.phuongThucThanhToan = $pttt,
            h.ngayTao = $ngayTao,
            h.tongTien = $tongTien

        WITH h

        MATCH (k:KhachHang {
            maKhachHang:$maKH
        })

        MERGE (k)-[:DAT]->(h)

        WITH h

        MATCH (n:NhanVien {
            maNhanVien:$maNV
        })

        MERGE (n)-[:TAO]->(h)

        WITH h

        MATCH (c:CuaHang {
            maCuaHang:$maCH
        })

        MERGE (h)-[:TAI_CUA_HANG]->(c)

    """,

    maHD=row["MAHD"],
    maCH=row["MACH"],
    maNV=row["MANV"],
    maKH=row["MAKH"],
    pttt=row["PHUONGTHUCTHANHTOAN"],
    ngayTao=str(row["NGAYTAO"]),
    tongTien=row["TONGTIEN"]
    )

print("Import HoaDon thành công")

In [ ]:
df = pd.read_csv("ChiTietHoaDon_CH1.csv")

for _, row in df.iterrows():

    graph.run("""

        MATCH (h:HoaDon {
            maHoaDon:$maHD
        })

        MATCH (s:SanPham {
            maSanPham:$maSP
        })

        MERGE (h)-[r:CHUA]->(s)

        SET r.soLuong = $sl,
            r.donGia = $dg,
            r.thanhTien = $tt

    """,

    maHD=row["MAHD"],
    maSP=row["MASANPHAM"],
    sl=row["SOLUONG"],
    dg=row["DONGIA"],
    tt=row["THANHTIEN"])
    				
print("Import ChiTietHoaDon thành công")

In [ ]:
CREATE INDEX hoaDon_index IF NOT EXISTS
FOR (h:HoaDon)
ON (h.maHoaDon)

In [ ]:
CREATE INDEX sanPham_index IF NOT EXISTS
FOR (s:SanPham)
ON (s.maSanPham)

In [ ]:
import pandas as pd
import time

query = """

UNWIND $rows AS row

MATCH (h:HoaDon {
    maHoaDon: row['MAHD']
})

MATCH (s:SanPham {
    maSanPham: row['MASANPHAM']
})

MERGE (h)-[r:CHUA]->(s)

SET r.soLuong = toInteger(row['SOLUONG']),
    r.donGia = toFloat(row['DONGIA']),
    r.thanhTien = toFloat(row['THANHTIEN'])

"""

chunks = pd.read_csv(
    "ChiTietHoaDon_CH1.csv",
    chunksize=1000
)

count = 0

start = time.time()

for chunk in chunks:

    records = chunk.to_dict("records")

    graph.run(query, rows=records)

    count += len(records)

    print("Đã import:", count)

end = time.time()

print("Hoàn tất")
print("Tổng thời gian:", end - start)

In [4]:
graph.run("""

CREATE INDEX hoaDon_index IF NOT EXISTS
FOR (h:HoaDon)
ON (h.maHoaDon)

""")

graph.run("""

CREATE INDEX sanPham_index IF NOT EXISTS
FOR (s:SanPham)
ON (s.maSanPham)

""")

(No data)

In [21]:
import pandas as pd

chunk_size = 50000

for i, chunk in enumerate(
    pd.read_csv("ChiTietHoaDon_CH1.csv", chunksize=chunk_size)
):

    filename = f"CTHD_{i+1}.csv"

    chunk.to_csv(
        filename,
        index=False
    )

    print("Đã tạo:", filename)

Đã tạo: CTHD_3_1.csv
Đã tạo: CTHD_3_2.csv


In [40]:
from py2neo import Graph
import pandas as pd

graph = Graph(
    "bolt://26.84.188.242:7687",
    auth=("neo4j", "123456789")
)

query = """

UNWIND $rows AS row

MATCH (h:HoaDon {
    maHoaDon: row['MAHD']
})

MATCH (s:SanPham {
    maSanPham: row['MASANPHAM']
})

MERGE (h)-[r:CHUA]->(s)

SET r.soLuong = toInteger(row['SOLUONG']),
    r.donGia = toFloat(row['DONGIA']),
    r.thanhTien = toFloat(row['THANHTIEN'])

"""

df = pd.read_csv("CTHD_20.csv")

batch_size = 1000

for i in range(0, len(df), batch_size):

    batch = df.iloc[i:i+batch_size]

    records = batch.to_dict("records")

    graph.run(query, rows=records)

    print(f"Đã import: {i + len(batch)}")

Đã import: 1000
Đã import: 2000
Đã import: 3000
Đã import: 4000
Đã import: 5000
Đã import: 6000
Đã import: 7000
Đã import: 8000
Đã import: 9000
Đã import: 10000
Đã import: 11000
Đã import: 12000
Đã import: 13000
Đã import: 14000
Đã import: 15000
Đã import: 16000
Đã import: 17000
Đã import: 18000
Đã import: 19000
Đã import: 20000
Đã import: 21000
Đã import: 22000
Đã import: 23000
Đã import: 24000
Đã import: 25000
Đã import: 26000
Đã import: 27000
Đã import: 28000
Đã import: 29000
Đã import: 30000
Đã import: 31000
Đã import: 32000
Đã import: 33000
Đã import: 34000
Đã import: 35000
Đã import: 36000
Đã import: 37000
Đã import: 38000
Đã import: 39000
Đã import: 40000
Đã import: 41000
Đã import: 42000
Đã import: 43000
Đã import: 44000
Đã import: 45000
Đã import: 46000
Đã import: 47000
Đã import: 48000
Đã import: 49000
Đã import: 49929


In [ ]:
import os

print(os.getcwd())